# 03. Señales reales: cargar, ver y escuchar audio, imágenes y señales biomédicas

**Objetivos de aprendizaje.** Al terminar podrás:

1. Cargar un archivo **.wav**, saber su $f_s$, número de muestras y duración, graficarlo con el **eje de tiempo correcto** y escucharlo.
2. Cargar una **imagen** y entenderla como una **señal en 2D**: una matriz $x[m, n]$.
3. Cargar una **señal biomédica** (ECG) desde un archivo de texto y reconocer sus partes y su ruido.
4. Comprobar que una señal "real" es exactamente lo mismo que las señales de los notebooks anteriores: **un arreglo de números**.

**Requisito:** notebooks 00-02. Los archivos de ejemplo están en la carpeta `datos/`.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. Audio

Un archivo .wav guarda: **la frecuencia de muestreo $f_s$** y **la lista de muestras**.

### Paso a paso: lo que hace `pds.cargar_audio` por dentro

In [ ]:
from scipy.io import wavfile

# Paso 1: leer el archivo -> nos da fs y las muestras "crudas"
fs, datos = wavfile.read("../datos/tono_440hz.wav")

print("fs =", fs, "muestras/segundo")
print("tipo de dato:", datos.dtype)          # int16 = enteros de 16 bits
print("forma:", datos.shape)                  # (N,) = mono   ;   (N, 2) = estéreo
print("primeras 8 muestras:", datos[:8])

Los .wav de 16 bits guardan **enteros** entre $-32768$ y $32767$. Para trabajar cómodamente los pasamos a
decimales entre $-1$ y $1$ dividiendo entre $32768 = 2^{15}$:

In [ ]:
# Paso 2: a decimales entre -1 y 1
x = datos / 32768.0
print("primeras 8 muestras:", np.round(x[:8], 4))

In [ ]:
# Paso 3: datos básicos de la señal
N = len(x)
duracion = N / fs
print(f"N = {N} muestras,  duración = N/fs = {duracion} s")

Todo esto (incluyendo convertir estéreo a mono) lo hace `pds.cargar_audio`:

In [ ]:
fs, x = pds.cargar_audio("../datos/tono_440hz.wav")

### Graficar con el eje de tiempo correcto

Si graficamos `x` sin más, el eje horizontal es la **posición** (número de muestra $n$). Para tener **segundos**
construimos $t = n / f_s$ (notebook 01).

In [ ]:
n = np.arange(0, len(x))
t = n / fs

figura, ejes = plt.subplots(2, 1, figsize=(10, 5))
pds.graficar_continua(t, x, titulo="Audio completo (1 s): se ve como un bloque", ax=ejes[0])

# acercamiento: los primeros 5 ms. ¿Cuántas muestras son?  n = t·fs
cuantas = pds.indice_de_muestra(0.005, fs)
pds.graficar_discreta(t[:cuantas], x[:cuantas], titulo=f"Primeros 5 ms = {cuantas} muestras: se ve que es una señal discreta",
                      etiqueta_x="t (s)", ax=ejes[1])
plt.tight_layout()
plt.show()

En 5 ms caben $440 \times 0.005 = 2.2$ ciclos. ¿Lo ves en la gráfica?

### Escuchar

In [ ]:
pds.reproducir(x, fs)

### Experimentos con audio

Como el audio es sólo un arreglo, podemos **operar** con él y escuchar el resultado.

In [ ]:
from IPython.display import display

print("1) Volumen a la mitad (escalar la amplitud)")
display(pds.reproducir(0.5 * x, fs))      # (reproducir normaliza; compara con guardar_audio)

print("2) Reproducir con fs al doble: dura la mitad y suena una octava más agudo")
display(pds.reproducir(x, 2 * fs))

print("3) Reproducir con fs a la mitad: dura el doble y suena una octava más grave")
display(pds.reproducir(x, fs // 2))

> **¿Por qué cambia el tono?** Las muestras son las mismas, así que la frecuencia **digital** $f$ no cambia.
> Pero al reproducir, la tarjeta de sonido convierte $f$ a Hz con $F = f \cdot f_s$: si $f_s$ se duplica, $F$ se duplica.

## 2. Imágenes: señales en 2D

Una imagen en escala de grises es una **matriz**: cada elemento es la intensidad de un pixel.
$$x[m, n], \quad m = \text{fila}, \quad n = \text{columna}$$
Una imagen a color tiene **3 matrices** (canales): rojo, verde y azul.

In [ ]:
imagen_color = pds.cargar_imagen("../datos/grace_hopper.jpg", en_grises=False)
print("forma:", imagen_color.shape, " = (filas, columnas, canales)")
print("pixel en fila 100, columna 200 (R, G, B):", imagen_color[100, 200, :])

In [ ]:
figura, ejes = plt.subplots(1, 4, figsize=(15, 4))
pds.mostrar_imagen(imagen_color, titulo="color", ax=ejes[0])
nombres = ["R (rojo)", "G (verde)", "B (azul)"]
for c in range(3):
    pds.mostrar_imagen(imagen_color[:, :, c], titulo=f"canal {nombres[c]}", ax=ejes[c + 1])
plt.tight_layout()
plt.show()

Para la mayor parte del curso usaremos **escala de grises** (una sola matriz):
$\text{gris} = 0.299R + 0.587G + 0.114B$.

In [ ]:
imagen = pds.cargar_imagen("../datos/grace_hopper.jpg")       # en_grises=True por defecto
pds.mostrar_imagen(imagen, titulo="escala de grises", barra_color=True)
plt.show()

### Un renglón de la imagen es una señal 1D

Si tomamos **una fila** de la matriz obtenemos una señal como las que ya conocemos: $x[n] = \text{imagen}[m_0, n]$.

In [ ]:
fila = 150
renglon = imagen[fila, :]                 # fila 150, todas las columnas
columnas = np.arange(0, len(renglon))

figura, ejes = plt.subplots(1, 2, figsize=(13, 4))
pds.mostrar_imagen(imagen, titulo=f"fila {fila} marcada", ax=ejes[0])
ejes[0].axhline(fila, color="r")
pds.graficar_continua(columnas, renglon, titulo=f"x[n] = imagen[{fila}, n]",
                      etiqueta_x="n (columna)", etiqueta_y="intensidad", ax=ejes[1])
plt.tight_layout()
plt.show()

### Operaciones con imágenes = operaciones con matrices

In [ ]:
negativo = 1 - imagen                      # invertir intensidades
recorte = imagen[50:250, 150:350]          # filas 50..249, columnas 150..349
espejo = imagen[:, ::-1]                   # invertir el orden de las columnas: es una inversión en el tiempo

figura, ejes = plt.subplots(1, 3, figsize=(14, 4))
pds.mostrar_imagen(negativo, titulo="negativo: 1 - x", ax=ejes[0])
pds.mostrar_imagen(recorte, titulo="recorte (slicing)", ax=ejes[1])
pds.mostrar_imagen(espejo, titulo="espejo: x[m, -n]", ax=ejes[2])
plt.tight_layout()
plt.show()

## 3. Señales biomédicas: electrocardiograma (ECG)

Las señales biomédicas suelen venir en **archivos de texto** (CSV): columnas de números. El archivo casi
nunca trae $f_s$: hay que buscarla en la documentación de la base de datos.

Nuestro archivo `ecg_sintetico.csv` tiene dos columnas: `tiempo_s, ecg_mV`, y fue adquirido a $f_s = 360$ Hz
(la misma frecuencia que la famosa base de datos *MIT-BIH Arrhythmia* de PhysioNet).

In [ ]:
# Miremos el archivo "crudo": las primeras líneas
with open("../datos/ecg_sintetico.csv") as archivo:
    for i in range(4):
        print(archivo.readline().strip())

In [ ]:
fs_ecg = 360
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)

In [ ]:
figura, ejes = plt.subplots(2, 1, figsize=(11, 6))
pds.graficar_continua(t_ecg, ecg, titulo="ECG completo (10 s)", etiqueta_y="mV", ax=ejes[0])

# acercamiento de 1.5 s a 3 s
inicio = pds.indice_de_muestra(1.5, fs_ecg)
fin = pds.indice_de_muestra(3.0, fs_ecg)
pds.graficar_continua(t_ecg[inicio:fin], ecg[inicio:fin], titulo="Acercamiento: 1.5 s a 3 s",
                      etiqueta_y="mV", ax=ejes[1])
plt.tight_layout()
plt.show()

**¿Qué observamos?**
- **Complejos QRS**: los picos altos y angostos (contracción de los ventrículos). Entre dos picos R hay un latido.
- **Onda P** (antes del QRS) y **onda T** (después).
- **Deriva de la línea base**: la señal sube y baja lentamente (respiración, movimiento de electrodos) → ruido de **baja frecuencia**.
- **Interferencia de 60 Hz** de la red eléctrica: la señal se ve "gruesa" → ruido de **alta frecuencia**.

En los notebooks 06 (filtros), 07 (correlación) y 08 (FFT) aprenderemos a **medir** y **quitar** esos ruidos.

> Para trabajar con señales reales: [PhysioNet](https://physionet.org) tiene cientos de bases de datos gratuitas
> (ECG, EEG, EMG...). Se pueden exportar a CSV desde su sitio o leerlas con el paquete `wfdb`.

## Comprueba tu comprensión

1. Un .wav tiene 240 000 muestras y $f_s = 48$ kHz. ¿Cuánto dura?
2. ¿Por qué un audio reproducido con el doble de $f_s$ suena más agudo?
3. En una imagen de 480 × 640, ¿qué representa `imagen[:, 100]`?

<details><summary>Ver respuestas</summary>

1. 5 s.
2. Porque las mismas muestras se reproducen en la mitad del tiempo: cada ciclo dura la mitad → $F = f\cdot f_s$ se duplica.
3. La **columna** 100 completa (480 valores): una señal 1D vertical.
</details>